In [2]:
import pandas as pd
import numpy as np

file_path = "C:/Users/asus/Desktop/Users/elliottisaac/OneDrive - Tulane University/Research_active/ssm_taxes/Data/restat_replication/Build/Input/acs_2012-2017.dat"

# Step 1: Read only 5 columns to find (year, serial) of same-sex couple households.
# These columns are sufficient to determine whether a person is in a same-sex couple.
colspecs_scan = [
    (0,   4),    # year
    (6,   14),   # serial
    (77,  81),   # pernum
    (106, 108),  # sploc
    (126, 130),  # related  (4-digit detailed version)
    (130, 131),  # sex
    (269, 270),  # qrelate
]
col_names_scan = ["year", "serial", "pernum", "sploc", "related", "sex", "qrelate"]

print("Step 1: scanning for same-sex couples...")
df_scan = pd.read_fwf(file_path, colspecs=colspecs_scan, names=col_names_scan)
print(f"  Total rows: {len(df_scan):,}")


Step 1: scanning for same-sex couples...
  Total rows: 18,871,967


In [3]:
# ── Identify same-sex couples using vectorised operations ──────────────────────────
# Core logic: find partner via sploc, then compare sex values.

# Sort for consistent merging
df_scan = df_scan.sort_values(["year","serial","pernum"]).reset_index(drop=True)

# Build a lookup table: (year, serial, pernum) → sex / related / qrelate
# Use merge instead of apply (much faster)
partner_info = df_scan[["year","serial","pernum","sex","related","qrelate"]].copy()
partner_info.columns = ["year","serial","sploc","partner_sex","partner_related","partner_qrelate"]

df_scan = df_scan.merge(partner_info, on=["year","serial","sploc"], how="left")

# Same-sex flag
same_sex    = df_scan["sex"] == df_scan["partner_sex"]
sploc_valid = df_scan["sploc"] > 0

# Same-sex married: head (101) + spouse (201), or vice versa
ss_mar = same_sex & sploc_valid & (
    ((df_scan["related"] == 101)  & (df_scan["partner_related"] == 201)) |
    ((df_scan["related"] == 201)  & (df_scan["partner_related"] == 101))
)

# Same-sex cohabiting: head (101) + unmarried partner (1114), or vice versa
ss_coh = same_sex & sploc_valid & (
    ((df_scan["related"] == 101)  & (df_scan["partner_related"] == 1114)) |
    ((df_scan["related"] == 1114) & (df_scan["partner_related"] == 101))
)

# mflag correction: qrelate==9 indicates the Census Bureau re-coded a same-sex
# spouse as "unmarried partner"; these should be treated as married.
mflag = (df_scan["related"] == 1114) & (df_scan["qrelate"] == 9) & ss_coh

# Propagate mflag to the partner as well
mflag_info = df_scan[["year","serial","pernum"]].copy()
mflag_info["mflag"] = mflag.values
mflag_info.columns = ["year","serial","sploc","partner_mflag"]
df_scan = df_scan.merge(mflag_info, on=["year","serial","sploc"], how="left")
df_scan["partner_mflag"] = df_scan["partner_mflag"].fillna(False)

ss_mar = ss_mar | mflag | df_scan["partner_mflag"]
ss_coh = ss_coh & ~mflag & ~df_scan["partner_mflag"]

df_scan["ss_all"] = (ss_mar | ss_coh).astype(int)

# Collect the set of (year, serial) keys that belong to SS households
ss_keys = df_scan.loc[df_scan["ss_all"] == 1, ["year","serial"]].drop_duplicates()
print(f"  Same-sex couple households found: {len(ss_keys):,}")
print(f"  Same-sex individuals: {df_scan['ss_all'].sum():,}")


C:\Users\asus\AppData\Local\Temp\ipykernel_1908\2326949204.py:39: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  df_scan["partner_mflag"] = df_scan["partner_mflag"].fillna(False)


  Same-sex couple households found: 54,027
  Same-sex individuals: 108,054


In [5]:
# Core modification: Add the educd (160, 163) field directly to avoid reading the full file twice.
colspecs_full = [
    (0, 4), (6, 14), (37, 39), (77, 81), (106, 108), (118, 119), 
    (124, 126), (126, 130), (130, 131), (131, 134), (134, 135), 
    (147, 148), (151, 152), (158, 160), (160, 163), (193, 194), 
    (251, 258), (261, 262), (269, 270)
]
col_names_full = [
    "year", "serial", "statefip", "pernum", "sploc", "nchild", 
    "relate", "related", "sex", "age", "marst", 
    "race", "hispan", "educ", "educd", "uhrswork", 
    "incearn", "migrate1", "qrelate"
]


# Create a hash set to speed up lookups
ss_set = set(zip(ss_keys["year"], ss_keys["serial"]))
chunks = []
CHUNK = 500_000

print("Step 2: reading full columns for SS households only...")
reader = pd.read_fwf(file_path, colspecs=colspecs_full, names=col_names_full, chunksize=CHUNK)

for chunk in reader:
    mask = [(y, s) in ss_set for y, s in zip(chunk["year"], chunk["serial"])]
    filtered = chunk[mask]
    if len(filtered) > 0:
        chunks.append(filtered)

df_full = pd.concat(chunks, ignore_index=True)


# Calculate the number of adults in the household (used for subsequent precise filtering of cohabiting couples).
df_full["adult"] = ((df_full["age"] >= 18) | (df_full["sploc"] != 0)).astype(int)
num_adults = df_full.groupby(["year","serial"])["adult"].sum().reset_index(name="num_adults")
df_full = df_full.merge(num_adults, on=["year","serial"], how="left")

print(f"Done. Full data for SS households: {len(df_full):,} rows")

Step 2: reading full columns for SS households only...
Done. Full data for SS households: 136,932 rows


In [11]:
# Sort data to ensure merge logic is accurate
df_clean = df_full.sort_values(["year","serial","pernum"]).reset_index(drop=True)

# Extract potential partner info and merge it back to the main table
p_info = df_clean[["year","serial","pernum","sex","related","qrelate","marst"]].copy()
p_info.columns = ["year","serial","sploc","p_sex","p_related","p_qrelate","p_marst"]
df_clean = df_clean.merge(p_info, on=["year","serial","sploc"], how="left")

# Basic conditions: Same sex and valid spouse location (sploc)
same_sex = df_clean["sex"] == df_clean["p_sex"]
sploc_valid = df_clean["sploc"] > 0

# Identify same-sex married (ss_mar) and same-sex cohabiting (ss_coh)
ss_mar = same_sex & sploc_valid & (((df_clean["related"] == 101) & (df_clean["p_related"] == 201)) | ((df_clean["related"] == 201) & (df_clean["p_related"] == 101)))
ss_coh = same_sex & sploc_valid & (((df_clean["related"] == 101) & (df_clean["p_related"] == 1114)) | ((df_clean["related"] == 1114) & (df_clean["p_related"] == 101)))

# Handle mflag (married couples identified by qrelate==9)
mflag = (df_clean["related"] == 1114) & (df_clean["qrelate"] == 9) & ss_coh
mflag_info = df_clean[["year","serial","pernum"]].copy()
mflag_info["mflag"] = mflag.values
mflag_info.columns = ["year","serial","sploc","p_mflag"]

df_clean = df_clean.merge(mflag_info, on=["year","serial","sploc"], how="left")
df_clean["p_mflag"] = df_clean["p_mflag"].fillna(False)


# Finalize marriage and cohabitation classification
ss_mar_final = ss_mar | mflag | df_clean["p_mflag"]
ss_coh_final = ss_coh & ~mflag & ~df_clean["p_mflag"]

df_clean["sscouple_mar"] = ss_mar_final.astype(int)
df_clean["sscouple_coh"] = ss_coh_final.astype(int)
df_clean["ss_any"] = (ss_mar_final | ss_coh_final).astype(int)

# Base filters: Age 18-60, exclude territories, exclude cases with allocated relationships
mask_base = (
    (df_clean["ss_any"] == 1) & 
    (df_clean["age"] >= 18) & (df_clean["age"] <= 60) & 
    (df_clean["statefip"] <= 56) & 
    (df_clean["qrelate"] != 4) & (df_clean.get("p_qrelate", 0) != 4)
)
df_valid = df_clean[mask_base].copy()

C:\Users\asus\AppData\Local\Temp\ipykernel_1908\3924121692.py:24: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  df_clean["p_mflag"] = df_clean["p_mflag"].fillna(False)


In [12]:
# ================= Assemble Samples =================
# Married logic: Head (101) and their spouse
mc = df_valid[df_valid["sscouple_mar"] == 1].copy()
head_m = mc[mc["related"] == 101]
spouse_m = mc[mc["related"].isin([201, 1114])]
spouse_m = spouse_m.sort_values("pernum").groupby(["year","serial"]).first().reset_index()
df_mar = head_m.merge(spouse_m, on=["year","serial"], suffixes=("_h","_s"))
df_mar["married"] = 1

In [13]:
# Cohabiting logic: To reach 21,136, we typically do not restrict num_adults
cc = df_valid[df_valid["sscouple_coh"] == 1].copy()
head_c = cc[cc["related"] == 101]
partner_c = cc[cc["related"] == 1114]
df_coh = head_c.merge(partner_c, on=["year","serial"], suffixes=("_h","_s"))
df_coh["married"] = 0

# Ensure bidirectional sploc matching (excludes complex cohabitation structures)
df_coh = df_coh[(df_coh["sploc_h"] == df_coh["pernum_s"]) & (df_coh["sploc_s"] == df_coh["pernum_h"])]

In [14]:
# Concatenate and drop duplicates
df_final = pd.concat([df_mar, df_coh], axis=0, ignore_index=True)
df_final = df_final.drop_duplicates(subset=["year", "serial"])

print(f"=== Final Sample (Target Alignment) ===")
print(f"Married:    {len(df_final[df_final['married']==1]):,} (Paper: 16,098)")
print(f"Cohabiting: {len(df_final[df_final['married']==0]):,} (Paper: 21,136)")

=== Final Sample (Target Alignment) ===
Married:    16,146 (Paper: 16,098)
Cohabiting: 21,761 (Paper: 21,136)


In [15]:
# Vectorized conversion: educd (detailed education code) to years of education
def educd_to_yrs_vec(s):
    s = s.fillna(-1).astype(int)
    result = np.full(len(s), np.nan)
    result[s <= 12] = 0
    result[s.isin([14, 15, 16, 17])] = s[s.isin([14, 15, 16, 17])] - 13
    result[s.isin([22, 23])] = s[s.isin([22, 23])] - 17
    result[s.isin([25, 26])] = s[s.isin([25, 26])] - 18
    result[s == 30] = 9
    result[s == 40] = 10
    result[s == 50] = 11
    result[(s >= 61) & (s <= 65)] = 12
    result[s == 71] = 13
    result[s == 81] = 14
    result[s == 101] = 16
    result[s >= 114] = 18
    return result

df_f = df_final.copy()





In [16]:
# Income and earnings split calculation
df_f["earn_h"] = df_f["incearn_h"].fillna(0).clip(lower=0)
df_f["earn_s"] = df_f["incearn_s"].fillna(0).clip(lower=0)
df_f["total_earn"] = df_f["earn_h"] + df_f["earn_s"]
df_f["earn_split"] = np.where(df_f["total_earn"] > 0, df_f[["earn_h","earn_s"]].max(axis=1) / df_f["total_earn"], np.nan)

# Construction of demographics and employment status variables
df_f["male"] = (df_f["sex_h"] == 1).astype(int)
df_f["female"] = (df_f["sex_h"] == 2).astype(int)
df_f["age_old"] = np.maximum(df_f["age_h"], df_f["age_s"])
df_f["age_yng"] = np.minimum(df_f["age_h"], df_f["age_s"])
df_f["age_diff"] = df_f["age_old"] - df_f["age_yng"]

df_f["both_work"] = ((df_f["earn_h"]>0) & (df_f["earn_s"]>0)).astype(int)
df_f["one_work"] = ((df_f["earn_h"]>0) ^ (df_f["earn_s"]>0)).astype(int)
df_f["none_work"] = ((df_f["earn_h"]==0) & (df_f["earn_s"]==0)).astype(int)

# Racial matching and dependent children identification
df_f["same_race"] = ((df_f["race_h"] == df_f["race_s"]) & ((df_f["hispan_h"] > 0) == (df_f["hispan_s"] > 0))).astype(int)
df_f["any_children"] = (df_f["nchild_h"] > 0).astype(int)

# Application of education years
df_f["edu_h_yrs"] = educd_to_yrs_vec(df_f["educd_h"])
df_f["edu_s_yrs"] = educd_to_yrs_vec(df_f["educd_s"])
df_f["edu_max"] = np.maximum(df_f["edu_h_yrs"], df_f["edu_s_yrs"])
df_f["edu_min"] = np.minimum(df_f["edu_h_yrs"], df_f["edu_s_yrs"])
df_f["edu_diff"] = df_f["edu_max"] - df_f["edu_min"]


# Output results comparison table
m = df_f[df_f["married"]==1]
c = df_f[df_f["married"]==0]

In [17]:
print("=== Table A2 Full Comparison ===\n")
print(f"{'Variable':<35} {'Your Married':>18} {'Your Cohab':>18} {'Paper Married':>14} {'Paper Cohab':>12}")
print("─"*100)

def row(label, s1, s2, pm, pc):
    v1 = f"{np.nanmean(s1):.3f} ({np.nanstd(s1):.3f})"
    v2 = f"{np.nanmean(s2):.3f} ({np.nanstd(s2):.3f})"
    print(f"{label:<35} {v1:>18} {v2:>18} {pm:>14} {pc:>12}")

row("Male", m["male"], c["male"], "0.469 (0.499)", "0.506 (0.500)")
row("Female", m["female"], c["female"], "0.531 (0.499)", "0.494 (0.500)")
row("Same race", m["same_race"], c["same_race"], "0.793 (0.405)", "0.757 (0.429)")
row("Age older", m["age_old"], c["age_old"], "46.205 (9.633)","43.353 (10.902)")
row("Age younger", m["age_yng"], c["age_yng"], "41.252 (9.831)","37.858 (10.442)")
row("Age diff", m["age_diff"], c["age_diff"], "4.953 (5.234)", "5.495 (5.543)")
row("Edu max (yrs)", m["edu_max"], c["edu_max"], "15.594 (2.468)","15.363 (2.264)")
row("Edu min (yrs)", m["edu_min"], c["edu_min"], "13.718 (3.044)","13.513 (2.604)")
row("Edu diff", m["edu_diff"], c["edu_diff"], "1.875 (2.286)", "1.850 (2.108)")
row("Any children", m["any_children"], c["any_children"], "0.314 (0.464)","0.162 (0.369)")
row("Both work", m["both_work"], c["both_work"], "0.776 (0.417)", "0.811 (0.392)")
row("Only 1 works", m["one_work"], c["one_work"], "0.195 (0.396)", "0.160 (0.367)")
row("Neither works", m["none_work"], c["none_work"], "0.029 (0.167)", "0.029 (0.168)")
row("Total earnings", m["total_earn"], c["total_earn"], "125287 (119780)","105188 (105192)")
row("Earnings split", m.loc[m["total_earn"]>0,"earn_split"], c.loc[c["total_earn"]>0,"earn_split"], "0.745 (0.200)", "0.723 (0.174)")
print("─"*100)
print(f"{'Observations':<35} {len(m):>18,} {len(c):>18,} {'16,098':>14} {'21,136':>12}")

=== Table A2 Full Comparison ===

Variable                                  Your Married         Your Cohab  Paper Married  Paper Cohab
────────────────────────────────────────────────────────────────────────────────────────────────────
Male                                     0.468 (0.499)      0.497 (0.500)  0.469 (0.499) 0.506 (0.500)
Female                                   0.532 (0.499)      0.503 (0.500)  0.531 (0.499) 0.494 (0.500)
Same race                                0.783 (0.412)      0.744 (0.436)  0.793 (0.405) 0.757 (0.429)
Age older                               46.209 (9.634)    43.251 (10.933) 46.205 (9.633) 43.353 (10.902)
Age younger                             41.241 (9.842)    37.698 (10.462) 41.252 (9.831) 37.858 (10.442)
Age diff                                 4.968 (5.263)      5.554 (5.690)  4.953 (5.234) 5.495 (5.543)
Edu max (yrs)                           15.588 (2.472)     15.345 (2.264) 15.594 (2.468) 15.363 (2.264)
Edu min (yrs)                        

In [ ]:
# ── Replicate Stata sscouple logic ──────────────────────────────────────
# Key: use sploc to locate the partner, then compare sex and related codes.

# Sort to match Stata ordering
df = df_full.sort_values(["year", "serial", "pernum"]).reset_index(drop=True)

# Build an index: (year, serial, pernum) → row position
df["_idx"] = df.index
lookup = df.set_index(["year", "serial", "pernum"])["_idx"].to_dict()

# Find the row index of each person's partner via sploc
def get_partner_idx(row):
    if row["sploc"] == 0:
        return -1
    return lookup.get((row["year"], row["serial"], row["sploc"]), -1)

print("Building partner index")
df["_partner_idx"] = df.apply(get_partner_idx, axis=1)


In [50]:
# ── Attach partner attributes to each row ────────────────────────────────
has_partner = df["_partner_idx"] >= 0
p_idx = df.loc[has_partner, "_partner_idx"].values

partner_sex     = np.full(len(df), np.nan)
partner_related = np.full(len(df), np.nan)
partner_qrelate = np.full(len(df), np.nan)

partner_sex[has_partner]     = df["sex"].values[p_idx]
partner_related[has_partner] = df["related"].values[p_idx]
partner_qrelate[has_partner] = df["qrelate"].values[p_idx]

df["partner_sex"]     = partner_sex
df["partner_related"] = partner_related
df["partner_qrelate"] = partner_qrelate

same_sex = df["sex"] == df["partner_sex"]
sploc_valid = df["sploc"] != 0

# sscouple_mar: head (101) paired with spouse (201), or vice versa, same sex
sscouple_mar = same_sex & sploc_valid & (
    ((df["related"] == 101) & (df["partner_related"] == 201)) |
    ((df["related"] == 201) & (df["partner_related"] == 101))
)

# sscouple_coh: head (101) paired with unmarried partner (1114), or vice versa, same sex
sscouple_coh = same_sex & sploc_valid & (
    ((df["related"] == 101)  & (df["partner_related"] == 1114)) |
    ((df["related"] == 1114) & (df["partner_related"] == 101))
)

# mflag: qrelate==9 on an unmarried partner (1114) in a SS cohabiting pair
# indicates the Census re-coded a same-sex spouse; treat as married.
mflag = (df["related"] == 1114) & (df["qrelate"] == 9) & sscouple_coh

# Propagate mflag to the partner
partner_mflag = np.full(len(df), False)
partner_mflag[has_partner] = mflag.values[p_idx]

# Apply correction: mflag individuals count as married, not cohabiting
sscouple_mar = sscouple_mar | mflag | partner_mflag
sscouple_coh = sscouple_coh & ~mflag & ~partner_mflag

df["sscouple_mar"] = sscouple_mar.astype(int)
df["sscouple_coh"] = sscouple_coh.astype(int)
df["sscouple_all"] = ((sscouple_mar | sscouple_coh)).astype(int)

print(f"sscouple_mar individuals: {df['sscouple_mar'].sum():,}")
print(f"sscouple_coh individuals: {df['sscouple_coh'].sum():,}")


sscouple_mar individuals: 51,940
sscouple_coh individuals: 56,114


In [51]:
# ── Filter: keep only same-sex couple members aged 18–60 ──────────────
df_ss = df[df["sscouple_all"] == 1].copy()
df_ss = df_ss[(df_ss["age"] >= 18) & (df_ss["age"] <= 60)].copy()

print(f"Same-sex couple members (18-60): {len(df_ss):,}")
print(f"  of which married: {df_ss['sscouple_mar'].sum():,}")
print(f"  of which cohab:   {df_ss['sscouple_coh'].sum():,}")


Same-sex couple members (18-60): 83,191
  of which married: 35,573
  of which cohab:   47,618


In [52]:
# ── Re-run identification on df_full, adding marst as supplementary criterion ──

df2 = df_full.copy()
df2 = df2.sort_values(["year","serial","pernum"]).reset_index(drop=True)

# Re-merge partner info (same as before, but now with full columns available)
partner_info = df2[["year","serial","pernum",
                     "sex","related","qrelate","marst","age"]].copy()
partner_info.columns = ["year","serial","sploc",
                        "partner_sex","partner_related",
                        "partner_qrelate","partner_marst","partner_age"]

df2 = df2.merge(partner_info, on=["year","serial","sploc"], how="left")

same_sex    = df2["sex"] == df2["partner_sex"]
sploc_valid = df2["sploc"] > 0

# ── Original criteria ────────────────────────────────────────────────────
ss_mar_orig = same_sex & sploc_valid & (
    ((df2["related"] == 101)  & (df2["partner_related"] == 201)) |
    ((df2["related"] == 201)  & (df2["partner_related"] == 101))
)
ss_coh_orig = same_sex & sploc_valid & (
    ((df2["related"] == 101)  & (df2["partner_related"] == 1114)) |
    ((df2["related"] == 1114) & (df2["partner_related"] == 101))
)

# ── mflag correction ─────────────────────────────────────────────────────
mflag = (df2["related"] == 1114) & (df2["qrelate"] == 9) & ss_coh_orig

mflag_info = df2[["year","serial","pernum"]].copy()
mflag_info["mflag"] = mflag.values
mflag_info.columns = ["year","serial","sploc","partner_mflag"]
df2 = df2.merge(mflag_info, on=["year","serial","sploc"], how="left")
df2["partner_mflag"] = df2["partner_mflag"].fillna(False)

ss_mar = ss_mar_orig | mflag | df2["partner_mflag"]
ss_coh = ss_coh_orig & ~mflag & ~df2["partner_mflag"]

# ── Supplementary criterion: use marst==1 for both partners ─────────────
# ACS 2012+ includes the ssmc variable for same-sex married couples;
# if ssmc is not available, use marst==1 for both partners as a fallback.
ss_mar_backup = (
    same_sex & sploc_valid &
    (df2["marst"] == 1) & (df2["partner_marst"] == 1) &
    ~ss_coh  # exclude cohabiting pairs
)
ss_mar = ss_mar | ss_mar_backup

df2["sscouple_mar"] = ss_mar.astype(int)
df2["sscouple_coh"] = ss_coh.astype(int)
df2["sscouple_all"] = (ss_mar | ss_coh).astype(int)

print(f"sscouple_mar individuals: {df2['sscouple_mar'].sum():,}")
print(f"sscouple_coh individuals: {df2['sscouple_coh'].sum():,}")


sscouple_mar individuals: 52,048
sscouple_coh individuals: 56,114


C:\Users\asus\AppData\Local\Temp\ipykernel_22032\1524882265.py:35: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  df2["partner_mflag"] = df2["partner_mflag"].fillna(False)


In [53]:
# ── Rebuild couple-level dataset ─────────────────────────────────────────
df_ss2 = df2[df2["sscouple_all"] == 1].copy()
df_ss2 = df_ss2[(df_ss2["age"] >= 18) & (df_ss2["age"] <= 60)].copy()

# Married: head + spouse, or marst==1 same-sex pairs
head_m    = df_ss2[(df_ss2["related"] == 101)  & (df_ss2["sscouple_mar"] == 1)]
spouse_m  = df_ss2[(df_ss2["related"] == 201)  & (df_ss2["sscouple_mar"] == 1)]
# Some married couple heads have a partner whose related code is not 201;
# supplement spouse pool with marst==1 individuals
other_m   = df_ss2[
    (df_ss2["sscouple_mar"] == 1) &
    (df_ss2["related"] != 101) &
    (df_ss2["related"] != 201)
]

mc = head_m.merge(
    pd.concat([spouse_m, other_m]),
    on=["year","serial"], suffixes=("_h","_s")
)
mc["married"] = 1

# Cohabiting: head + unmarried partner
head_c    = df_ss2[(df_ss2["related"] == 101)  & (df_ss2["sscouple_coh"] == 1)]
partner_c = df_ss2[(df_ss2["related"] == 1114) & (df_ss2["sscouple_coh"] == 1)]

cc = head_c.merge(
    partner_c,
    on=["year","serial"], suffixes=("_h","_s")
)
cc["married"] = 0

df_couple2 = pd.concat([mc, cc], axis=0, ignore_index=True)
df_couple2 = df_couple2[
    (df_couple2["age_h"] >= 18) & (df_couple2["age_h"] <= 60) &
    (df_couple2["age_s"] >= 18) & (df_couple2["age_s"] <= 60)
].copy()

print(f"Couple-level: {len(df_couple2):,}")
print(f"  Married:    {df_couple2['married'].sum():,}")
print(f"  Cohabiting: {(df_couple2['married']==0).sum():,}")


Couple-level: 38,345
  Married:    16,213
  Cohabiting: 22,132


In [54]:
# Diagnostic: inspect one duplicated household
example = df_ss2[
    (df_ss2["year"] == 2012) & (df_ss2["serial"] == 62643)
][["year","serial","pernum","sploc","related","sex","age","marst","qrelate","sscouple_mar","sscouple_coh"]]
print(example.to_string())


     year  serial  pernum  sploc  related  sex  age  marst  qrelate  sscouple_mar  sscouple_coh
769  2012   62643       1      7      101    1   25      6        0             1             0
770  2012   62643       2      3      701    2   24      1        0             1             0
771  2012   62643       3      2      801    2   24      1        3             1             0
775  2012   62643       7      1     1114    1   27      4        9             1             0


In [55]:
# ── Fix: deduplicate married pairs; leave cohabiting unchanged ──────────
#
# Duplicates in mc arise because other_m contains non-201 individuals
# in addition to the true spouse.
# Solution: build married pairs using head + spouse (related==201) only,
# then supplement with ssmc/marst for spouses whose related code is not 201.

head_m   = df_ss2[(df_ss2["related"] == 101) & (df_ss2["sscouple_mar"] == 1)].copy()
spouse_m = df_ss2[(df_ss2["sscouple_mar"] == 1) & (df_ss2["related"] != 101)].copy()

# On the spouse side, keep only the person with the lowest pernum per household
spouse_m = (spouse_m
            .sort_values("pernum")
            .groupby(["year","serial"])
            .first()
            .reset_index())

mc2 = head_m.merge(spouse_m, on=["year","serial"], suffixes=("_h","_s"))
mc2["married"] = 1

# Cohabiting unchanged
cc2 = cc.copy()

df_couple3 = pd.concat([mc2, cc2], axis=0, ignore_index=True)

# Age filter
df_couple3 = df_couple3[
    (df_couple3["age_h"] >= 18) & (df_couple3["age_h"] <= 60) &
    (df_couple3["age_s"] >= 18) & (df_couple3["age_s"] <= 60)
].copy()

# Final deduplication (should be empty, but kept as a safety check)
df_couple3 = df_couple3.drop_duplicates(subset=["year","serial"])

print(f"Couple-level: {len(df_couple3):,}")
print(f"  Married:    {df_couple3['married'].sum():,}")
print(f"  Cohabiting: {(df_couple3['married']==0).sum():,}")


Couple-level: 38,281
  Married:    16,149
  Cohabiting: 22,132


In [56]:
# ── Diagnostic: distribution of adult household members in cohabiting sample ──

# Count adults per household in df_full
df_full["adult"] = ((df_full["age"] >= 18) | (df_full["sploc"] != 0)).astype(int)
num_adults = df_full.groupby(["year","serial"])["adult"].sum().reset_index()
num_adults.columns = ["year","serial","num_adults"]

# Merge adult count into cohabiting sample
cc_check = cc3.merge(num_adults, on=["year","serial"], how="left")
print("num_adults distribution in cohabiting sample:")
print(cc_check["num_adults"].value_counts().sort_index().head(10))


num_adults distribution in cohabiting sample:
num_adults
2     18793
3      2453
4       660
5       169
6        36
7         9
8         5
9         2
11        3
12        2
Name: count, dtype: int64


In [57]:
# ── Fix married sample: drop extra pairs introduced by ss_mar_backup ────────
# Married couples should be identified solely via related==101/201 + mflag;
# the marst-based backup criterion is removed here.

df3 = df_full.copy()
df3 = df3.sort_values(["year","serial","pernum"]).reset_index(drop=True)

partner_info2 = df3[["year","serial","pernum","sex","related","qrelate","marst"]].copy()
partner_info2.columns = ["year","serial","sploc","partner_sex","partner_related","partner_qrelate","partner_marst"]
df3 = df3.merge(partner_info2, on=["year","serial","sploc"], how="left")

same_sex2    = df3["sex"] == df3["partner_sex"]
sploc_valid2 = df3["sploc"] > 0

ss_mar2 = same_sex2 & sploc_valid2 & (
    ((df3["related"] == 101)  & (df3["partner_related"] == 201)) |
    ((df3["related"] == 201)  & (df3["partner_related"] == 101))
)
ss_coh2 = same_sex2 & sploc_valid2 & (
    ((df3["related"] == 101)  & (df3["partner_related"] == 1114)) |
    ((df3["related"] == 1114) & (df3["partner_related"] == 101))
)

mflag2 = (df3["related"] == 1114) & (df3["qrelate"] == 9) & ss_coh2

mflag_info2 = df3[["year","serial","pernum"]].copy()
mflag_info2["mflag"] = mflag2.values
mflag_info2.columns = ["year","serial","sploc","partner_mflag"]
df3 = df3.merge(mflag_info2, on=["year","serial","sploc"], how="left")
df3["partner_mflag"] = df3["partner_mflag"].fillna(False)

ss_mar2 = ss_mar2 | mflag2 | df3["partner_mflag"]
ss_coh2 = ss_coh2 & ~mflag2 & ~df3["partner_mflag"]

df3["sscouple_mar"] = ss_mar2.astype(int)
df3["sscouple_coh"] = ss_coh2.astype(int)
df3["sscouple_all"] = (ss_mar2 | ss_coh2).astype(int)

# Age filter
df_ss3 = df3[(df3["sscouple_all"]==1) & (df3["age"]>=18) & (df3["age"]<=60)].copy()

# Build married couple-level dataset
head_m2   = df_ss3[(df_ss3["related"]==101)  & (df_ss3["sscouple_mar"]==1)]
spouse_m2 = df_ss3[(df_ss3["related"]==201)  & (df_ss3["sscouple_mar"]==1)]

# mflag-corrected individuals: coded as related==1114 but treated as married
mflag_spouse = df_ss3[(df_ss3["sscouple_mar"]==1) & (df_ss3["related"]==1114)]

spouse_all = pd.concat([spouse_m2, mflag_spouse]).sort_values("pernum")
spouse_all = spouse_all.groupby(["year","serial"]).first().reset_index()

mc4 = head_m2.merge(spouse_all, on=["year","serial"], suffixes=("_h","_s"))
mc4["married"] = 1
mc4 = mc4.drop_duplicates(subset=["year","serial"])

print(f"Married: {len(mc4):,}")
print(f"Cohabiting: {len(cc4):,}")


Married: 16,147
Cohabiting: 18,793


C:\Users\asus\AppData\Local\Temp\ipykernel_22032\2538867008.py:30: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  df3["partner_mflag"] = df3["partner_mflag"].fillna(False)


In [58]:
# ── Final merge ─────────────────────────────────────────────────────────
df_final = pd.concat([mc4, cc4], axis=0, ignore_index=True)
df_final = df_final[
    (df_final["age_h"] >= 18) & (df_final["age_h"] <= 60) &
    (df_final["age_s"] >= 18) & (df_final["age_s"] <= 60)
].copy()
df_final = df_final.drop_duplicates(subset=["year","serial"])

print(f"\n=== Final Sample ===")
print(f"Total:      {len(df_final):,}")
print(f"Married:    {df_final['married'].sum():,}")
print(f"Cohabiting: {(df_final['married']==0).sum():,}")



=== Final Sample ===
Total:      34,940
Married:    16,147
Cohabiting: 18,793


In [59]:
import numpy as np

# Use the final sample with qrelate==4 removed
mc_f = mc4[(mc4["qrelate_h"] != 4) & (mc4["qrelate_s"] != 4)].copy()
cc_f = cc3[(cc3["qrelate_h"] != 4) & (cc3["qrelate_s"] != 4)].copy()

df_f = pd.concat([mc_f, cc_f], ignore_index=True)

# Construct basic variables
df_f["earn_h"] = df_f["incearn_h"].fillna(0).clip(lower=0)
df_f["earn_s"] = df_f["incearn_s"].fillna(0).clip(lower=0)
df_f["total_earn"] = df_f["earn_h"] + df_f["earn_s"]
df_f["earn_split"] = np.where(
    df_f["total_earn"] > 0,
    df_f[["earn_h","earn_s"]].max(axis=1) / df_f["total_earn"],
    np.nan
)

df_f["male"]     = (df_f["sex_h"] == 1).astype(int)
df_f["female"]   = (df_f["sex_h"] == 2).astype(int)
df_f["age_old"]  = np.maximum(df_f["age_h"], df_f["age_s"])
df_f["age_yng"]  = np.minimum(df_f["age_h"], df_f["age_s"])
df_f["age_diff"] = df_f["age_old"] - df_f["age_yng"]

df_f["both_work"] = ((df_f["earn_h"]>0) & (df_f["earn_s"]>0)).astype(int)
df_f["one_work"]  = ((df_f["earn_h"]>0) ^ (df_f["earn_s"]>0)).astype(int)
df_f["none_work"] = ((df_f["earn_h"]==0) & (df_f["earn_s"]==0)).astype(int)

# Same race: race codes match for both partners
df_f["same_race"] = (df_f["race_h"] == df_f["race_s"]).astype(int)

m = df_f[df_f["married"]==1]
c = df_f[df_f["married"]==0]

def row(label, s1, s2, paper_m=None, paper_c=None):
    v1, v2 = np.nanmean(s1), np.nanmean(s2)
    sd1, sd2 = np.nanstd(s1), np.nanstd(s2)
    paper = f"  |  paper: {paper_m} / {paper_c}" if paper_m else ""
    print(f"{label:<35} {v1:.3f} ({sd1:.3f})   {v2:.3f} ({sd2:.3f}){paper}")

print(f"{'Variable':<35} {'Married':>18}   {'Cohabiting':>18}   | Paper targets")
print("─"*90)
row("Male",              m["male"],      c["male"],      "0.469","0.506")
row("Female",            m["female"],    c["female"],    "0.531","0.494")
row("Same race",         m["same_race"], c["same_race"], "0.793","0.757")
row("Age older partner", m["age_old"],   c["age_old"],   "46.21","43.35")
row("Age younger partner",m["age_yng"],  c["age_yng"],   "41.25","37.86")
row("Age difference",    m["age_diff"],  c["age_diff"],  "4.95", "5.50")
row("Both work",         m["both_work"], c["both_work"], "0.776","0.811")
row("Only 1 works",      m["one_work"],  c["one_work"],  "0.195","0.160")
row("Neither works",     m["none_work"], c["none_work"], "0.029","0.029")
row("Total earnings",    m["total_earn"],c["total_earn"],"125287","105188")
row("Earnings split",
    m.loc[m["total_earn"]>0,"earn_split"],
    c.loc[c["total_earn"]>0,"earn_split"], "0.745","0.723")
print("─"*90)
print(f"{'Observations':<35} {len(m):>8,}              {len(c):>8,}")


Variable                                       Married           Cohabiting   | Paper targets
──────────────────────────────────────────────────────────────────────────────────────────
Male                                0.468 (0.499)   0.497 (0.500)  |  paper: 0.469 / 0.506
Female                              0.532 (0.499)   0.503 (0.500)  |  paper: 0.531 / 0.494
Same race                           0.855 (0.352)   0.828 (0.377)  |  paper: 0.793 / 0.757
Age older partner                   46.209 (9.634)   43.251 (10.933)  |  paper: 46.21 / 43.35
Age younger partner                 41.241 (9.842)   37.698 (10.462)  |  paper: 41.25 / 37.86
Age difference                      4.968 (5.263)   5.554 (5.690)  |  paper: 4.95 / 5.50
Both work                           0.776 (0.417)   0.812 (0.391)  |  paper: 0.776 / 0.811
Only 1 works                        0.195 (0.396)   0.159 (0.366)  |  paper: 0.195 / 0.160
Neither works                       0.029 (0.168)   0.029 (0.167)  |  paper: 0.029 

In [60]:
# Paper definition of same race: race codes match AND Hispanic status matches.
# Hispanic: hispan > 0 → Hispanic; hispan == 0 → non-Hispanic.
df_f["hisp_h"] = (df_f["hispan_h"] > 0).astype(int)
df_f["hisp_s"] = (df_f["hispan_s"] > 0).astype(int)

df_f["same_race"] = (
    (df_f["race_h"] == df_f["race_s"]) &
    (df_f["hisp_h"] == df_f["hisp_s"])
).astype(int)

m = df_f[df_f["married"]==1]
c = df_f[df_f["married"]==0]

print(f"Same race - Married: {m['same_race'].mean():.3f}, Cohabiting: {c['same_race'].mean():.3f}")
print(f"Paper:               Married: 0.793,          Cohabiting: 0.757")


Same race - Married: 0.783, Cohabiting: 0.744
Paper:               Married: 0.793,          Cohabiting: 0.757


In [61]:
# ── Children: use the nchild variable ────────────────────────────────────
# nchild = number of own children in household.
# The paper uses c_anychildren (any children) and c_children (count).

# df_f contains nchild_h and nchild_s;
# at the couple level, use the head's nchild (which already covers all own children).
df_f["any_children"] = (df_f["nchild_h"] > 0).astype(int)
df_f["num_children"] = df_f["nchild_h"]

m = df_f[df_f["married"]==1]
c = df_f[df_f["married"]==0]

print("Any dependent children:")
print(f"  Married:    {m['any_children'].mean():.3f} ({m['any_children'].std():.3f})")
print(f"  Cohabiting: {c['any_children'].mean():.3f} ({c['any_children'].std():.3f})")
print(f"  Paper:      Married=0.314, Cohabiting=0.162")

print("\nConditional number of children (among those with any):")
print(f"  Married:    {m.loc[m['any_children']==1,'num_children'].mean():.3f}")
print(f"  Cohabiting: {c.loc[c['any_children']==1,'num_children'].mean():.3f}")
print(f"  Paper:      Married=1.813, Cohabiting=1.694")


Any dependent children:
  Married:    0.334 (0.472)
  Cohabiting: 0.176 (0.381)
  Paper:      Married=0.314, Cohabiting=0.162

Conditional number of children (among those with any):
  Married:    1.804
  Cohabiting: 1.663
  Paper:      Married=1.813, Cohabiting=1.694


In [62]:
# Re-read educd (bytes 161–163, 0-indexed: 160–163) from the raw file.
# Only the educd column needs to be added; no need to re-read everything.

colspecs_educd = [
    (0,   4),    # year
    (6,   14),   # serial
    (77,  81),   # pernum
    (160, 163),  # educd (bytes 161–163)
]
col_names_educd = ["year", "serial", "pernum", "educd"]

# Read in chunks, keeping only SS households
print("Reading educd...")
chunks_edu = []
reader_edu = pd.read_fwf(
    file_path,
    colspecs=colspecs_educd,
    names=col_names_educd,
    chunksize=500_000
)
for chunk in reader_edu:
    mask = [(y,s) in ss_set for y,s in zip(chunk["year"], chunk["serial"])]
    filtered = chunk[mask]
    if len(filtered) > 0:
        chunks_edu.append(filtered)

df_educd = pd.concat(chunks_edu, ignore_index=True)
print(f"Done: {len(df_educd):,} rows")


Reading educd...


KeyboardInterrupt: 

In [ ]:
# Merge educd into df_f.
# Head columns are suffixed _h; spouse columns are suffixed _s.
# Look up educd separately for each partner using pernum.

educd_lookup = df_educd.set_index(["year","serial","pernum"])["educd"]

# Vectorised lookup via apply (df_f is small after couple-level reshape)
df_f["educd_h"] = df_f.apply(
    lambda r: educd_lookup.get((r["year"], r["serial"], r["pernum_h"]), np.nan), axis=1
)
df_f["educd_s"] = df_f.apply(
    lambda r: educd_lookup.get((r["year"], r["serial"], r["pernum_s"]), np.nan), axis=1
)

print("educd_h sample:", df_f["educd_h"].value_counts().head(8))


educd_h sample: educd_h
101    10697
114     6231
71      5790
63      4241
81      3378
65      2318
115     1755
116     1203
Name: count, dtype: int64


In [ ]:
# Vectorised conversion: educd → years of education
def educd_to_yrs_vec(s):
    s = s.fillna(-1).astype(int)
    result = np.full(len(s), np.nan)
    result[s <= 12]               = 0
    result[s == 14]               = 1
    result[s == 15]               = 2
    result[s == 16]               = 3
    result[s == 17]               = 4
    result[s == 22]               = 5
    result[s == 23]               = 6
    result[s == 25]               = 7
    result[s == 26]               = 8
    result[s == 30]               = 9
    result[s == 40]               = 10
    result[s == 50]               = 11
    result[(s >= 61) & (s <= 65)] = 12
    result[s == 71]               = 13
    result[s == 81]               = 14
    result[s == 101]              = 16
    result[s >= 114]              = 18
    result[s == -1]               = np.nan
    return result

df_f["edu_h_yrs"] = educd_to_yrs_vec(df_f["educd_h"])
df_f["edu_s_yrs"] = educd_to_yrs_vec(df_f["educd_s"])

df_f["edu_max"]  = np.maximum(df_f["edu_h_yrs"], df_f["edu_s_yrs"])
df_f["edu_min"]  = np.minimum(df_f["edu_h_yrs"], df_f["edu_s_yrs"])
df_f["edu_diff"] = df_f["edu_max"] - df_f["edu_min"]

m = df_f[df_f["married"]==1]
c = df_f[df_f["married"]==0]

print("=== Table A2 Full Comparison ===\n")
print(f"{'Variable':<35} {'Your Married':>18} {'Your Cohab':>18} {'Paper Married':>14} {'Paper Cohab':>12}")
print("─"*100)

def row(label, s1, s2, pm, pc):
    v1 = f"{np.nanmean(s1):.3f} ({np.nanstd(s1):.3f})"
    v2 = f"{np.nanmean(s2):.3f} ({np.nanstd(s2):.3f})"
    print(f"{label:<35} {v1:>18} {v2:>18} {pm:>14} {pc:>12}")

row("Male",              m["male"],       c["male"],       "0.469 (0.499)", "0.506 (0.500)")
row("Female",            m["female"],     c["female"],     "0.531 (0.499)", "0.494 (0.500)")
row("Same race",         m["same_race"],  c["same_race"],  "0.793 (0.405)", "0.757 (0.429)")
row("Age older",         m["age_old"],    c["age_old"],    "46.205 (9.633)","43.353 (10.902)")
row("Age younger",       m["age_yng"],    c["age_yng"],    "41.252 (9.831)","37.858 (10.442)")
row("Age diff",          m["age_diff"],   c["age_diff"],   "4.953 (5.234)", "5.495 (5.543)")
row("Edu max (yrs)",     m["edu_max"],    c["edu_max"],    "15.594 (2.468)","15.363 (2.264)")
row("Edu min (yrs)",     m["edu_min"],    c["edu_min"],    "13.718 (3.044)","13.513 (2.604)")
row("Edu diff",          m["edu_diff"],   c["edu_diff"],   "1.875 (2.286)", "1.850 (2.108)")
row("Any children",      m["any_children"],c["any_children"],"0.314 (0.464)","0.162 (0.369)")
row("Both work",         m["both_work"],  c["both_work"],  "0.776 (0.417)", "0.811 (0.392)")
row("Only 1 works",      m["one_work"],   c["one_work"],   "0.195 (0.396)", "0.160 (0.367)")
row("Neither works",     m["none_work"],  c["none_work"],  "0.029 (0.167)", "0.029 (0.168)")
row("Total earnings",    m["total_earn"], c["total_earn"], "125287 (119780)","105188 (105192)")
row("Earnings split",
    m.loc[m["total_earn"]>0,"earn_split"],
    c.loc[c["total_earn"]>0,"earn_split"], "0.745 (0.200)", "0.723 (0.174)")

print("─"*100)
print(f"{'Observations':<35} {len(m):>18,} {len(c):>18,} {'16,098':>14} {'21,136':>12}")

# Sanity check: null counts after educd conversion
print(f"\nedu_max null: {df_f['edu_max'].isna().sum()}")
print(f"edu_min null: {df_f['edu_min'].isna().sum()}")


=== Table A2 完整对比 ===

变量                                           你的Married            你的Cohab      论文Married      论文Cohab
────────────────────────────────────────────────────────────────────────────────────────────────────
Male                                     0.468 (0.499)      0.497 (0.500)  0.469 (0.499) 0.506 (0.500)
Female                                   0.532 (0.499)      0.503 (0.500)  0.531 (0.499) 0.494 (0.500)
Same race                                0.855 (0.352)      0.828 (0.377)  0.793 (0.405) 0.757 (0.429)
Age older                               46.209 (9.634)    43.251 (10.933) 46.205 (9.633) 43.353 (10.902)
Age younger                             41.241 (9.842)    37.698 (10.462) 41.252 (9.831) 37.858 (10.442)
Age diff                                 4.968 (5.263)      5.554 (5.690)  4.953 (5.234) 5.495 (5.543)
Edu max (yrs)                           15.588 (2.472)     15.345 (2.264) 15.594 (2.468) 15.363 (2.264)
Edu min (yrs)                           13.708 (